[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-12b-quantization-from-scratch.ipynb)

# Advanced Discussion 12 (continued) — absmax, zero-point, groups, GPTQ and AWQ

Quantisers written from scratch, the outlier problem, a quantised MLP, a simplified GPTQ and an AWQ-style scaling sweep.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Uses scikit-learn's bundled digits; runs in seconds.

In [ ]:
%pip install -q torch scikit-learn numpy

In [ ]:
import numpy as np, torch
torch.manual_seed(0); np.random.seed(0)

## 1. the two basic schemes on one weight vector: absmax (symmetric) and zero-point (asymmetric)

In [ ]:
def q_absmax(w, bits=8):
    qmax = 2 ** (bits - 1) - 1; scale = w.abs().max() / qmax
    q = torch.round(w / scale).clamp(-qmax - 1, qmax); return q * scale
def q_zeropoint(w, bits=8):
    lo, hi = w.min(), w.max(); levels = 2 ** bits - 1; scale = (hi - lo) / levels; zp = torch.round(-lo / scale)
    q = torch.round(w / scale + zp).clamp(0, levels); return (q - zp) * scale
w = torch.randn(4096) * 0.02
print("weights ~ N(0, 0.02^2), 4096 values; max |w| = %.4f" % w.abs().max())
for bits in (8, 4):
    print("  %d-bit: absmax mean abs error %.6f | zero-point %.6f" % (bits, (q_absmax(w, bits) - w).abs().mean(), (q_zeropoint(w, bits) - w).abs().mean()))
w_skew = torch.randn(4096) * 0.02 + 0.03                                        # all values shifted: an asymmetric range
print("  shifted (asymmetric) weights, 4-bit: absmax %.6f | zero-point %.6f" % ((q_absmax(w_skew, 4) - w_skew).abs().mean(), (q_zeropoint(w_skew, 4) - w_skew).abs().mean()))

## 2. one outlier ruins a per-tensor scale; finer scale groups contain the damage

In [ ]:
def q_group(w, bits, group):
    flat = w.reshape(-1, group); qmax = 2 ** (bits - 1) - 1; scale = flat.abs().amax(1, keepdim=True) / qmax
    return (torch.round(flat / scale).clamp(-qmax - 1, qmax) * scale).reshape(w.shape)
w = torch.randn(4096) * 0.02; w_out = w.clone(); w_out[100] = 1.0                    # a single outlier, 50x the typical weight
print("\n4-bit mean abs error with one outlier (|w|=1.0) among 4096 weights:")
print("  per-tensor absmax                 %.6f (without the outlier: %.6f)" % ((q_absmax(w_out, 4) - w_out).abs().mean(), (q_absmax(w, 4) - w).abs().mean()))
for g in (1024, 128, 32):
    print("  groups of %-4d (one scale each)    %.6f" % (g, (q_group(w_out, 4, g) - w_out).abs().mean()))
print("  scale storage: a group of 128 with one fp16 scale costs 16/128 = %.3f extra bits per weight" % (16 / 128))

## 3. a real (small) network: train an MLP on digits, then quantize its weights (round to nearest)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
X, y = load_digits(return_X_y=True); X = X / 16.0
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
Xtr, Xte = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(Xte, dtype=torch.float32); ytr, yte = torch.tensor(ytr), torch.tensor(yte)
net = torch.nn.Sequential(torch.nn.Linear(64, 128), torch.nn.ReLU(), torch.nn.Linear(128, 64), torch.nn.ReLU(), torch.nn.Linear(64, 10))
opt = torch.optim.Adam(net.parameters(), 3e-3)
for ep in range(120):
    opt.zero_grad(); loss = torch.nn.functional.cross_entropy(net(Xtr), ytr); loss.backward(); opt.step()
acc = lambda m: (m(Xte).argmax(1) == yte).float().mean().item()
base = acc(net); print("\ntrained MLP (64-128-64-10) test accuracy: %.3f; weights %d params -> %.1f KB in fp32" % (base, sum(p.numel() for p in net.parameters()), 4 * sum(p.numel() for p in net.parameters()) / 1024))
import copy
def quantized(fn):
    m = copy.deepcopy(net)
    with torch.no_grad():
        for mod in m:
            if isinstance(mod, torch.nn.Linear): mod.weight.copy_(fn(mod.weight))
    return m
print("bits   per-tensor absmax   group-of-16 absmax   (accuracy)")
for bits in (8, 6, 4, 3, 2):
    pt = acc(quantized(lambda w: q_absmax(w, bits))); gp = acc(quantized(lambda w: q_group(w, bits, 16)))
    print("  %d       %.3f               %.3f" % (bits, pt, gp))

## 4. GPTQ idea on one layer: round one column at a time and push the error onto the columns not yet rounded

In [ ]:
def rtn(W, bits, group=16):
    return q_group(W, bits, group)
def gptq(W, H, bits, group=16, damp=0.01):
    """Simplified GPTQ: process input columns in order; after rounding column j, spread its error onto later columns using the inverse Hessian H = 2 X^T X."""
    W = W.clone(); n = W.shape[1]; H = H.clone(); H += damp * torch.diag(H).mean() * torch.eye(n)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(torch.linalg.cholesky(H)), upper=True); Q = torch.zeros_like(W)
    qmax = 2 ** (bits - 1) - 1
    for j in range(n):
        if j % group == 0: scale = W[:, j:j + group].abs().amax(1, keepdim=True) / qmax
        col = W[:, j:j + 1]; q = torch.round(col / scale).clamp(-qmax - 1, qmax) * scale; Q[:, j:j + 1] = q
        err = (col - q) / Hinv[j, j]; W[:, j + 1:] -= err @ Hinv[j:j + 1, j + 1:]
    return Q
# a layer with correlated inputs, as real activations are
torch.manual_seed(1); n_in, n_out, N = 128, 64, 2048
mix = torch.randn(n_in, n_in) * 0.3 + torch.eye(n_in); X = torch.randn(N, n_in) @ mix
W = torch.randn(n_out, n_in) * 0.05
H = 2 * X.T @ X / N
print("\nlayer 128 -> 64 with correlated inputs; error is the mean squared difference of the layer OUTPUT (lower is better)")
print("bits   round-to-nearest   GPTQ-style   reduction")
for bits in (4, 3, 2):
    e_r = ((X @ rtn(W, bits).T - X @ W.T) ** 2).mean().item(); e_g = ((X @ gptq(W, H, bits).T - X @ W.T) ** 2).mean().item()
    print("  %d      %.5f            %.5f       %.0f%%" % (bits, e_r, e_g, 100 * (1 - e_g / e_r)))

## 5. AWQ idea: a few input channels carry large activations; protect their weights by scaling

In [ ]:
torch.manual_seed(2); X2 = torch.randn(N, n_in); salient = torch.randperm(n_in)[:4]; X2[:, salient] *= 12.0
W2 = torch.randn(n_out, n_in) * 0.05
def awq_like(W, X, bits, alpha):
    s = X.abs().mean(0) ** alpha; s = s / s.mean()                                   # per-input-channel scale from activation magnitude
    return q_group(W * s, bits, 16) / s
print("\n4-bit, 4 of 128 input channels have 12x larger activations")
for alpha in (0.0, 0.25, 0.5, 0.75, 1.0):
    Wq = awq_like(W2, X2, 4, alpha); print("  scale exponent %.2f   output error %.4f" % (alpha, ((X2 @ Wq.T - X2 @ W2.T) ** 2).mean().item()))

## 6. memory arithmetic

In [ ]:
print("\nweights only, bytes per parameter -> GB")
for name, n in (("7B", 7e9), ("13B", 13e9), ("70B", 70e9)):
    print("  %-4s fp16 %.1f GB | int8 %.1f GB | 4-bit with group-128 fp16 scales %.1f GB" % (name, n * 2 / 1e9, n * 1 / 1e9, n * (0.5 + 2 / 128) / 1e9))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.